In [ ]:
# ─────────────────────────────────────────────────────────
# 
# Reads the cleaned CSV and loads data into 6 PostgreSQL tables.
# Run this AFTER:
#   1. PostgreSQL is running (brew services start postgresql@16)
#   2. Tables are created (01_create_tables.sql has been run)
#   3. EDA notebook has been run (cleaned CSV exists)
#
# Usage: python load_to_postgres.py
# ─────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# ─────────────────────────────────────────
# CONFIG — update username if needed
# ─────────────────────────────────────────
DB_USER = "vittoriobariosco"  # your macOS username
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "clinical_trials" # your PostgreSQL database name 
CSV_PATH = "/Users/vittoriobariosco/Documents/APPLICATIONS/MIGx/data/processed/clinical_trials_clean.csv" 

# path to cleaned CSV from EDA notebook

# ─────────────────────────────────────────
# 1. CONNECT TO POSTGRESQL
# ─────────────────────────────────────────
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database()"))
    print(f"✅ Connected to: {result.fetchone()[0]}")

# ─────────────────────────────────────────
# 2. LOAD CLEANED CSV
# ─────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f"✅ Loaded CSV: {len(df):,} rows × {len(df.columns)} columns")

# Parse dates
for col in ['Start Date', 'Completion Date', 'Primary Completion Date']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# ─────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────
def split_pipe(value):
    """Split pipe-separated string into list. Handle NaN."""
    if pd.isna(value):
        return []
    return [x.strip() for x in str(value).split('|') if x.strip()]

def normalize_condition(condition):
    """Normalize condition name variants to canonical names."""
    c = condition.lower().strip()
    if any(t in c for t in [
        'covid', 'sars-cov', 'sars cov', 'sars cov-2',
        'coronavirus', 'corona virus', '2019-ncov',
        'ncov', 'cytokine storm', 'cytokine release', 'anosmia'
    ]):
        return 'COVID-19'
    elif any(t in c for t in ['ards', 'acute respiratory distress']):
        return 'Acute Respiratory Distress Syndrome'
    elif 'pneumonia' in c:
        return 'Pneumonia'
    elif any(t in c for t in ['acute kidney injury', ' aki']):
        return 'Acute Kidney Injury'
    else:
        return condition.strip().title()

def assign_therapeutic_area(condition):
    """Assign therapeutic area category to normalized condition."""
    c = condition.lower().strip()
    if any(t in c for t in [
        'covid', 'sars', 'coronavirus', '2019-ncov',
        'viral infection', 'virus disease', 'infection viral',
        'influenza', 'sepsis', 'inflammation', 'infection',
        'hiv', 'aids', 'inflammatory response', 'epidemic'
    ]):
        return 'Infectious Disease'
    elif any(t in c for t in [
        'pneumonia', 'respiratory', 'ards', 'lung', 'pulmonary',
        'hypoxemia', 'hypoxia', 'dyspnea', 'asthma', 'copd',
        'mechanical ventilation', 'critical illness', 'cytokine',
        'post intensive care', 'intensive care unit'
    ]):
        return 'Respiratory/Critical Care'
    elif any(t in c for t in [
        'anxiety', 'depression', 'mental', 'psychiatric',
        'ptsd', 'post traumatic', 'stress', 'psychological',
        'social isolation', 'isolation, social', 'loneliness',
        'depressive', 'burnout', 'wellbeing', 'well-being',
        'sleep', 'insomnia'
    ]):
        return 'Mental Health'
    elif any(t in c for t in [
        'diabetes', 'obesity', 'cardiovascular', 'heart',
        'hypertension', 'thrombosis', 'stroke', 'vitamin d',
        'metabolic', 'myocardial', 'myocarditis',
        'endothelial', 'cardiac', 'coagulation', 'thromboembolism'
    ]):
        return 'Metabolic/Cardiovascular'
    elif any(t in c for t in [
        'cancer', 'tumor', 'oncology', 'carcinoma',
        'lymphoma', 'neoplasm', 'leukemia', 'malignant'
    ]):
        return 'Oncology'
    elif any(t in c for t in [
        'multiple sclerosis', 'parkinson', 'autism',
        'neurolog', 'dementia', 'alzheimer', 'epilepsy',
        'cognitive impairment', 'cognitive'
    ]):
        return 'Neurological'
    elif any(t in c for t in [
        'kidney', 'renal', 'aki', 'acute kidney'
    ]):
        return 'Renal'
    elif any(t in c for t in [
        'arthritis', 'rheumatoid', 'lupus', 'musculoskeletal',
        'pain', 'chronic pain', 'rheumatic'
    ]):
        return 'Musculoskeletal/Rheumatology'
    elif any(t in c for t in [
        'pregnancy', 'maternal', 'obstetric',
        'neonatal', 'perinatal', 'prenatal'
    ]):
        return 'Reproductive Health'
    else:
        return 'Other'

def parse_intervention(intervention_string):
    """Parse 'Drug: Remdesivir' into type and name."""
    if ':' in intervention_string:
        parts = intervention_string.split(':', 1)
        return {
            'intervention_type': parts[0].strip(),
            'name': parts[1].strip()
        }
    return {
        'intervention_type': 'Other',
        'name': intervention_string.strip()
    }

def parse_study_design(design_string):
    """Parse 'Allocation: Randomized|Masking: Double' into dict."""
    result = {
        'allocation': None,
        'intervention_model': None,
        'masking': None,
        'primary_purpose': None,
        'observational_model': None,
        'time_perspective': None
    }
    if pd.isna(design_string):
        return result
    for part in str(design_string).split('|'):
        if ':' in part:
            key, val = part.split(':', 1)
            key = key.strip().lower()
            val = val.strip()
            if 'allocation' in key:
                result['allocation'] = val
            elif 'intervention model' in key:
                result['intervention_model'] = val
            elif 'masking' in key:
                result['masking'] = val
            elif 'primary purpose' in key:
                result['primary_purpose'] = val
            elif 'observational model' in key:
                result['observational_model'] = val
            elif 'time perspective' in key:
                result['time_perspective'] = val
    return result

continent_map = {
    # North America
    'United States': 'North America', 'Canada': 'North America',
    'Mexico': 'North America', 'Puerto Rico': 'North America',
    'Costa Rica': 'North America', 'Panama': 'North America',
    'Dominican Republic': 'North America', 'Honduras': 'North America',
    'Guatemala': 'North America', 'Jamaica': 'North America',
    'Martinique': 'North America', 'Guadeloupe': 'North America',
    'Barbados': 'North America',
    # South America
    'Brazil': 'South America', 'Argentina': 'South America',
    'Colombia': 'South America', 'Chile': 'South America',
    'Peru': 'South America', 'Ecuador': 'South America',
    'Bolivia': 'South America', 'Paraguay': 'South America',
    'Uruguay': 'South America', 'Venezuela': 'South America',
    'French Guiana': 'South America',
    # Europe
    'France': 'Europe', 'Spain': 'Europe', 'Italy': 'Europe',
    'United Kingdom': 'Europe', 'Germany': 'Europe', 'Belgium': 'Europe',
    'Netherlands': 'Europe', 'Denmark': 'Europe', 'Poland': 'Europe',
    'Greece': 'Europe', 'Switzerland': 'Europe', 'Sweden': 'Europe',
    'Austria': 'Europe', 'Ukraine': 'Europe',
    'Russian Federation': 'Europe', 'Norway': 'Europe',
    'Portugal': 'Europe', 'Finland': 'Europe', 'Hungary': 'Europe',
    'Romania': 'Europe', 'Czechia': 'Europe', 'Bulgaria': 'Europe',
    'Serbia': 'Europe', 'Croatia': 'Europe', 'Slovakia': 'Europe',
    'Ireland': 'Europe', 'Georgia': 'Europe', 'Belarus': 'Europe',
    'Lithuania': 'Europe', 'Bosnia and Herzegovina': 'Europe',
    'Latvia': 'Europe', 'Slovenia': 'Europe', 'Estonia': 'Europe',
    'North Macedonia': 'Europe', 'Albania': 'Europe',
    'Luxembourg': 'Europe', 'Monaco': 'Europe', 'Iceland': 'Europe',
    'Cyprus': 'Europe', 'Malta': 'Europe', 'San Marino': 'Europe',
    # Asia
    'China': 'Asia', 'Japan': 'Asia', 'India': 'Asia',
    'Turkey': 'Asia', 'Israel': 'Asia', 'Iran': 'Asia',
    'South Korea': 'Asia', 'Saudi Arabia': 'Asia', 'Pakistan': 'Asia',
    'Taiwan': 'Asia', 'Singapore': 'Asia', 'Indonesia': 'Asia',
    'Hong Kong': 'Asia', 'Malaysia': 'Asia', 'Philippines': 'Asia',
    'Bangladesh': 'Asia', 'Jordan': 'Asia', 'Nepal': 'Asia',
    'United Arab Emirates': 'Asia', 'Qatar': 'Asia', 'Thailand': 'Asia',
    'Vietnam': 'Asia', 'Kuwait': 'Asia', 'Iraq': 'Asia',
    'Lebanon': 'Asia', 'Kazakhstan': 'Asia', 'Bahrain': 'Asia',
    'Uzbekistan': 'Asia', 'Cambodia': 'Asia', 'Oman': 'Asia',
    'Azerbaijan': 'Asia', 'Armenia': 'Asia', 'Mongolia': 'Asia',
    'Kyrgyzstan': 'Asia', "Lao People's Democratic Republic": 'Asia',
    # Africa
    'Egypt': 'Africa', 'South Africa': 'Africa', 'Nigeria': 'Africa',
    'Ethiopia': 'Africa', 'Kenya': 'Africa', 'Tunisia': 'Africa',
    'Zimbabwe': 'Africa', 'Zambia': 'Africa', 'Uganda': 'Africa',
    'Ghana': 'Africa', 'Mali': 'Africa', 'Malawi': 'Africa',
    'Mozambique': 'Africa', 'Burkina Faso': 'Africa',
    'Senegal': 'Africa', 'Cameroon': 'Africa', 'Sudan': 'Africa',
    'Guinea-Bissau': 'Africa', 'Gambia': 'Africa',
    "Côte D'Ivoire": 'Africa', 'Guinea': 'Africa',
    'Cape Verde': 'Africa', 'Algeria': 'Africa', 'Botswana': 'Africa',
    'Tanzania': 'Africa', 'South Sudan': 'Africa', 'Rwanda': 'Africa',
    'Lesotho': 'Africa', 'Mayotte': 'Africa', 'Réunion': 'Africa',
    # Oceania
    'Australia': 'Oceania', 'New Zealand': 'Oceania',
    'New Caledonia': 'Oceania',
}

def parse_location(loc_string):
    """Extract country and continent from location string."""
    parts = [p.strip() for p in loc_string.split(',')]
    country = parts[-1].strip()
    return {
        'country': country,
        'continent': continent_map.get(country, 'Other')
    }

# ─────────────────────────────────────────
# 3. LOAD STUDIES TABLE
# ─────────────────────────────────────────
print("\n Loading studies table ")

studies_df = pd.DataFrame({
    'nct_id':                  df['NCT Number'],
    'title':                   df['Title'],
    'acronym':                 df['Acronym'],
    'status':                  df['Status'],
    'status_group':            df['status_group'],
    'phase':                   df['phase_clean'],
    'study_type':              df['Study Type'],
    'start_date':              df['Start Date'],
    'completion_date':         df['Completion Date'],
    'primary_completion_date': df['Primary Completion Date'],
    'enrollment':              df['Enrollment'],
    'gender':                  df['Gender'],
    'pre_covid_start':         df['pre_covid_start'],
    'date_error':              df['date_error'],
})

studies_df = studies_df.drop_duplicates(subset='nct_id')
studies_df.to_sql('studies', engine, if_exists='append', index=False)
print(f"   loaded {len(studies_df):,} rows loaded into studies")

# ─────────────────────────────────────────
# 4. GET study_id MAPPING
# ─────────────────────────────────────────
print("\n Fetching study_id mapping from PostgreSQL")

with engine.connect() as conn:
    id_map_df = pd.read_sql("SELECT study_id, nct_id FROM studies", conn)

id_map = dict(zip(id_map_df['nct_id'], id_map_df['study_id']))
print(f"   loaded {len(id_map):,} study IDs")

# ─────────────────────────────────────────
# 5. LOAD CONDITIONS TABLE
# ─────────────────────────────────────────
print("\n Loading conditions table ")

conditions_rows = []
for _, row in df.iterrows():
    study_id = id_map.get(row['NCT Number'])
    if not study_id:
        continue
    for condition in split_pipe(row['Conditions']):
        normalized = normalize_condition(condition)
        conditions_rows.append({
            'study_id':         study_id,
            'condition_name':   normalized,
            'mesh_term':        None,
            'therapeutic_area': assign_therapeutic_area(normalized)
        })

conditions_df = pd.DataFrame(conditions_rows)
conditions_df.to_sql('conditions', engine, if_exists='append', index=False)
print(f"   {len(conditions_df):,} rows loaded into conditions")

# ─────────────────────────────────────────
# 6. LOAD INTERVENTIONS TABLE
# ─────────────────────────────────────────
print("\n Loading interventions table ")

interventions_rows = []
for _, row in df.iterrows():
    study_id = id_map.get(row['NCT Number'])
    if not study_id:
        continue
    for intervention in split_pipe(row['Interventions']):
        parsed = parse_intervention(intervention)
        interventions_rows.append({
            'study_id':          study_id,
            'intervention_type': parsed['intervention_type'],
            'name':              parsed['name'],
            'description':       None
        })

interventions_df = pd.DataFrame(interventions_rows)
interventions_df.to_sql('interventions', engine, if_exists='append', index=False)
print(f"   loaded {len(interventions_df):,} rows loaded into interventions")

# ─────────────────────────────────────────
# 7. LOAD SPONSORS TABLE
# ─────────────────────────────────────────
print("\n Loading sponsors table ")

sponsors_rows = []
for _, row in df.iterrows():
    study_id = id_map.get(row['NCT Number'])
    if not study_id:
        continue
    sponsors = split_pipe(row['Sponsor/Collaborators'])
    funded_bys = split_pipe(row['Funded Bys'])
    for i, sponsor in enumerate(sponsors):
        sponsors_rows.append({
            'study_id':             study_id,
            'agency':               sponsor,
            'agency_class':         funded_bys[i] if i < len(funded_bys) else None,
            'lead_or_collaborator': 'Lead' if i == 0 else 'Collaborator'
        })

sponsors_df = pd.DataFrame(sponsors_rows)
sponsors_df.to_sql('sponsors', engine, if_exists='append', index=False)
print(f"    {len(sponsors_df):,} rows loaded into sponsors")

# ─────────────────────────────────────────
# 7.5 LOAD OUTCOMES TABLE
# ─────────────────────────────────────────
print("\n Loading outcomes table ")

outcomes_rows = []
for _, row in df.iterrows():
    study_id = id_map.get(row['NCT Number'])
    if not study_id:
        continue
    for outcome in split_pipe(row['Outcome Measures']):
        outcomes_rows.append({
            'study_id':    study_id,
            'outcome_type': None,      # not parseable from this dataset
            'measure':     outcome,
            'time_frame':  None,
            'description': None
        })

outcomes_df = pd.DataFrame(outcomes_rows)
outcomes_df.to_sql('outcomes', engine, if_exists='append', index=False)
print(f"   {len(outcomes_df):,} rows loaded into outcomes")

# ─────────────────────────────────────────
# 8. LOAD LOCATIONS TABLE
# ─────────────────────────────────────────
print("\n Loading locations table ")

locations_rows = []
for _, row in df.iterrows():
    study_id = id_map.get(row['NCT Number'])
    if not study_id:
        continue
    for location in split_pipe(row['Locations']):
        parsed = parse_location(location)
        locations_rows.append({
            'study_id':  study_id,
            'facility':  None,
            'city':      None,
            'state':     None,
            'country':   parsed['country'],
            'continent': parsed['continent']
        })

locations_df = pd.DataFrame(locations_rows)
locations_df.to_sql('locations', engine, if_exists='append', index=False)
print(f"   ✅ {len(locations_df):,} rows loaded into locations")

# ─────────────────────────────────────────
# 9. LOAD STUDY DESIGN TABLE
# ─────────────────────────────────────────
print("\n Loading study_design table ")

design_rows = []
for _, row in df.iterrows():
    study_id = id_map.get(row['NCT Number'])
    if not study_id:
        continue
    parsed = parse_study_design(row['Study Designs'])
    parsed['study_id'] = study_id
    design_rows.append(parsed)

design_df = pd.DataFrame(design_rows)
design_df.to_sql('study_design', engine, if_exists='append', index=False)
print(f"   {len(design_df):,} rows loaded into study_design")

# ─────────────────────────────────────────
# 10. FINAL VERIFICATION
# ─────────────────────────────────────────
print("\n📊 FINAL VERIFICATION — Row counts per table:")
tables = ['studies', 'conditions', 'interventions',
          'sponsors', 'locations', 'study_design', 'outcomes']

with engine.connect() as conn:
    for table in tables:
        count = conn.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        ).fetchone()[0]
        print(f"   {table:<20} {count:>8,} rows")

print("\n🎉 All tables loaded successfully!")